# DriveSense-VLM — 00: Data Pipeline

**GPU**: T4 (or CPU) | **Time**: ~2–3 h | **Cost**: ~5 CU

> ⚠️ **Use a FRESH runtime.** Never downgrade numpy — Colab ships numpy 2.x and all
> packages (pandas, opencv, jax, tensorflow) are compiled against numpy 2.x C ABI.
> Reinstalling numpy<2 causes `ValueError: numpy.dtype size changed` on every import.

Runs the complete data curation pipeline:
1. nuScenes mini download + rarity filtering (6-signal composite score)
2. PySpark distributed ETL + analytics
3. DADA-2000 critical-moment extraction (optional, 53 GB)
4. Unified dataset assembly (stratified train/val/test splits)
5. LLM counterfactual annotation (mock = free | real ≈ $5–8 via Anthropic API)
6. SFT JSONL formatting with correct image paths for Qwen3-VL training

> **RECOVERY**: If Colab disconnects, rerun Cells 2–3 (setup + install),
> then continue from the interrupted section. Every step skips if its output exists.

In [ ]:
# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ── Paths ──
PROJECT_ROOT = "/content/drive/MyDrive/DriveSense-VLM"
REPO_ROOT    = "/content/DriveSense-VLM"
DATA_ROOT    = f"{PROJECT_ROOT}/data"
OUTPUTS_ROOT = f"{PROJECT_ROOT}/outputs"

# Create Drive directories
for d in [DATA_ROOT, f"{DATA_ROOT}/nuscenes", f"{DATA_ROOT}/dada2000",
          OUTPUTS_ROOT, f"{OUTPUTS_ROOT}/data", f"{OUTPUTS_ROOT}/training",
          f"{OUTPUTS_ROOT}/data/sft_ready"]:
    os.makedirs(d, exist_ok=True)

# Clone repo to fast ephemeral SSD
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/jayanth922/DriveSense-VLM.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
os.chdir(REPO_ROOT)
print(f"Working dir: {os.getcwd()}")

# Symlink data/outputs → Drive (persistent across disconnects)
!ln -sfn {DATA_ROOT} {REPO_ROOT}/data
!ln -sfn {OUTPUTS_ROOT} {REPO_ROOT}/outputs

# Add src to Python path (replaces broken editable install)
sys.path.insert(0, f"{REPO_ROOT}/src")
print(f"✓ Repo:  {REPO_ROOT}")
print(f"✓ Drive: {PROJECT_ROOT}")

In [ ]:
# Install data-pipeline dependencies
# ⚠️  NEVER install numpy or pandas — Colab's pre-installed versions are compiled
#     against numpy 2.x ABI. Reinstalling them breaks pandas, opencv, jax, etc.
# nuscenes-devkit pins numpy<2; install with --no-deps then add real deps manually.
!pip install --upgrade pip setuptools wheel -q 2>&1 | tail -1
!pip install nuscenes-devkit --no-deps -q 2>&1 | tail -1
!pip install pyquaternion matplotlib tqdm Pillow pyyaml requests openpyxl -q 2>&1 | tail -1
!pip install scikit-learn scipy pyspark -q 2>&1 | tail -1

sys.path.insert(0, f"{REPO_ROOT}/src")

# Verify — numpy/pandas are Colab's pre-installed versions (numpy ~2.x)
import numpy as np, pandas as pd, drivesense
print(f"✓ numpy {np.__version__} | pandas {pd.__version__} | DriveSense loaded")
from nuscenes.nuscenes import NuScenes
print("✓ nuscenes-devkit importable")

## 1. nuScenes Mini Download

nuScenes mini is ~1 GB and sufficient for a full pipeline test.

**Option A** (automatic): The cell below tries `wget` from the official URL.
**Option B** (manual): Upload `v1.0-mini.tgz` to your Drive and run the fallback cell.

nuScenes registration: https://www.nuscenes.org/nuscenes#download

In [ ]:
import os

NUSCENES_DIR = f"{DATA_ROOT}/nuscenes"
MINI_DIR     = f"{NUSCENES_DIR}/v1.0-mini"
SAMPLES_DIR  = f"{NUSCENES_DIR}/samples"

if os.path.exists(MINI_DIR) and os.path.exists(SAMPLES_DIR):
    print("✓ nuScenes mini already extracted — skipping download.")
else:
    os.makedirs(NUSCENES_DIR, exist_ok=True)
    print("Downloading nuScenes mini (~1 GB)...")
    !wget -q "https://www.nuscenes.org/data/v1.0-mini.tgz" \
          -O /tmp/nuscenes_mini.tgz || true

    tgz = "/tmp/nuscenes_mini.tgz"
    if os.path.exists(tgz) and os.path.getsize(tgz) > 1_000_000:
        print("Extracting...")
        !tar -xzf {tgz} -C {NUSCENES_DIR}
        !rm {tgz}
        print(f"✓ Extracted to {NUSCENES_DIR}")
    else:
        print("⚠ Auto-download failed (URL may require authentication).")
        print("  Upload v1.0-mini.tgz to Drive and run the fallback cell below.")

In [ ]:
# FALLBACK: Extract from manually uploaded archive on Drive
import os, glob

NUSCENES_DIR = f"{DATA_ROOT}/nuscenes"
already_done = (os.path.exists(f"{NUSCENES_DIR}/v1.0-mini") and
                os.path.exists(f"{NUSCENES_DIR}/samples"))

if already_done:
    print("✓ nuScenes already extracted — skipping.")
else:
    search_paths = [PROJECT_ROOT, "/content/drive/MyDrive"]
    candidates = []
    for base in search_paths:
        candidates += glob.glob(f"{base}/**/*.tgz", recursive=True)
    tgz = next((c for c in candidates
                if any(k in c.lower() for k in ("nuscenes", "mini", "v1.0"))), None)

    if tgz:
        print(f"Found: {tgz}")
        os.makedirs(NUSCENES_DIR, exist_ok=True)
        !tar -xzf {tgz} -C {NUSCENES_DIR}
        print(f"✓ Extracted to {NUSCENES_DIR}")
    else:
        print(f"⚠ No archive found. Upload v1.0-mini.tgz to {PROJECT_ROOT}/ then rerun.")

In [ ]:
# Verify nuScenes loads correctly
import os
from nuscenes.nuscenes import NuScenes

NUSCENES_DIR = f"{DATA_ROOT}/nuscenes"
print(f"Contents of {NUSCENES_DIR}:")
!ls {NUSCENES_DIR} 2>/dev/null || echo "  (directory empty or missing)"

if os.path.exists(f"{NUSCENES_DIR}/v1.0-mini"):
    try:
        nusc = NuScenes(version="v1.0-mini", dataroot=NUSCENES_DIR, verbose=False)
        print(f"\n✓ nuScenes v1.0-mini loaded:")
        print(f"  Scenes      : {len(nusc.scene)}")
        print(f"  Samples     : {len(nusc.sample)}")
        print(f"  Annotations : {len(nusc.sample_annotation)}")
    except Exception as e:
        print(f"\n✗ Load failed: {e}")
        print(f"  Expected layout inside {NUSCENES_DIR}/: v1.0-mini/  samples/  maps/")
else:
    print("\n⚠ v1.0-mini/ not found — complete the download step first.")

## 2. Rarity Filtering

Scores each frame on 6 binary signals (proximity, occlusion, density, adverse weather,
VRU at intersection, cyclist). Keeps frames with composite score ≥ threshold.
Use `--min-score 1` for the mini dataset to retain enough frames.

**Output**: `metadata.json` (JSON array) → immediately converted to `metadata.jsonl` (JSON Lines).
The unified builder's `load_nuscenes_frames()` requires JSONL format.

In [ ]:
import os, json

FILTER_OUTPUT = f"{OUTPUTS_ROOT}/data/nuscenes_filtered"
NUSCENES_DIR  = f"{DATA_ROOT}/nuscenes"
os.makedirs(FILTER_OUTPUT, exist_ok=True)

json_path  = f"{FILTER_OUTPUT}/metadata.json"
jsonl_path = f"{FILTER_OUTPUT}/metadata.jsonl"

if os.path.exists(jsonl_path):
    with open(jsonl_path) as f:
        n = sum(1 for _ in f)
    print(f"✓ Already filtered ({n} frames) — skipping.")
elif os.path.exists(json_path):
    print("metadata.json found — converting to JSONL (filter already ran)...")
else:
    print("Running nuScenes rarity filter...")
    !python scripts/run_nuscenes_filter.py \
        --nuscenes-root {NUSCENES_DIR} \
        --version v1.0-mini \
        --output-dir {FILTER_OUTPUT} \
        --min-score 1 \
        2>&1

# ── CRITICAL: Convert JSON array → JSONL ────────────────────────────────────
# run_nuscenes_filter.py writes metadata.json (array).
# UnifiedDatasetBuilder.load_nuscenes_frames() expects metadata.jsonl (JSON Lines).
if os.path.exists(json_path) and not os.path.exists(jsonl_path):
    with open(json_path) as f:
        frames = json.load(f)
    with open(jsonl_path, "w") as f:
        for frame in frames:
            f.write(json.dumps(frame) + "\n")
    print(f"  ✓ Converted {len(frames)} frames → metadata.jsonl")

if os.path.exists(jsonl_path):
    with open(jsonl_path) as f:
        n = sum(1 for _ in f)
    print(f"  ✓ metadata.jsonl: {n} frames")
else:
    print("  ⚠ metadata.json not produced — check filter output above")

!ls -lh {FILTER_OUTPUT} 2>/dev/null

## 3. PySpark ETL

Distributed rarity scoring with signal co-occurrence analytics and temporal clustering.
Runs in local Spark mode — no cluster needed.

In [ ]:
import os

SPARK_OUTPUT = f"{OUTPUTS_ROOT}/data/spark_processed"

if os.path.exists(SPARK_OUTPUT) and os.listdir(SPARK_OUTPUT):
    print(f"✓ Spark output already exists — skipping.")
    !ls {SPARK_OUTPUT}
else:
    os.makedirs(SPARK_OUTPUT, exist_ok=True)
    os.environ.setdefault("JAVA_HOME", "/usr/lib/jvm/java-11-openjdk-amd64")
    print("Running PySpark rarity scoring pipeline...")
    !python scripts/run_spark_pipeline.py \
        --version v1.0-mini \
        --nuscenes-root {DATA_ROOT}/nuscenes \
        --output-dir {SPARK_OUTPUT} \
        2>&1
    print("✓ Spark pipeline complete.")
    !ls {SPARK_OUTPUT}

## 4. DADA-2000 (Optional)

DADA-2000 is a 53 GB accident dashcam dataset. **Skip for a mini/demo run.**

To include it:
1. Download from https://github.com/JWFangit/LOTVS-DADA
2. Upload to Drive: `DriveSense-VLM/data/dada2000/DADA-2000/`
3. Structure: `DADA-2000/<category>/<sequence>/images/`
4. Also upload `dada_text_annotations.xlsx` to `data/dada2000/`

In [ ]:
import os

DADA_ROOT   = f"{DATA_ROOT}/dada2000"
DADA_DATA   = f"{DADA_ROOT}/DADA-2000"
DADA_OUTPUT = f"{OUTPUTS_ROOT}/data/dada_extracted"

if os.path.exists(DADA_DATA) and os.listdir(DADA_DATA):
    print(f"✓ DADA-2000 found at {DADA_DATA}")
    if os.path.exists(f"{DADA_OUTPUT}/metadata.jsonl"):
        print("✓ Already extracted — skipping.")
    else:
        os.makedirs(DADA_OUTPUT, exist_ok=True)
        print("Running DADA-2000 extraction...")
        !python scripts/run_dada_extraction.py \
            --dada-root {DADA_ROOT} \
            --output-dir {DADA_OUTPUT} \
            2>&1
        print("✓ DADA-2000 extraction complete.")
        !ls {DADA_OUTPUT}
else:
    print("⚠ DADA-2000 not found — skipping (mini run mode).")
    print(f"  To enable: upload data to {DADA_DATA}/")

## 5. Unified Dataset

Merges nuScenes filtered frames + DADA-2000 (if present) into stratified
train/val/test splits. Outputs `train_manifest.jsonl`, `val_manifest.jsonl`,
`test_manifest.jsonl`. Then combines them into `all_manifest.jsonl` for annotation.

In [ ]:
import os, json

FILTER_OUTPUT  = f"{OUTPUTS_ROOT}/data/nuscenes_filtered"
DADA_OUTPUT    = f"{OUTPUTS_ROOT}/data/dada_extracted"
UNIFIED_OUTPUT = f"{OUTPUTS_ROOT}/data/unified"
os.makedirs(UNIFIED_OUTPUT, exist_ok=True)

if os.path.exists(f"{UNIFIED_OUTPUT}/train_manifest.jsonl"):
    print("✓ Unified dataset already built — skipping.")
    !ls {UNIFIED_OUTPUT}
else:
    HAS_DADA  = os.path.exists(f"{DADA_OUTPUT}/metadata.jsonl")
    dada_flag = f"--dada-dir {DADA_OUTPUT}" if HAS_DADA else "--nuscenes-only"
    src_desc  = "nuScenes + DADA-2000" if HAS_DADA else "nuScenes only"
    print(f"Building unified dataset ({src_desc})...")
    !python scripts/run_build_unified_dataset.py \
        --nuscenes-dir {FILTER_OUTPUT} \
        --output-dir {UNIFIED_OUTPUT} \
        {dada_flag} \
        2>&1

    print("\n✓ Split manifests:")
    for split in ("train", "val", "test"):
        p = f"{UNIFIED_OUTPUT}/{split}_manifest.jsonl"
        if os.path.exists(p):
            with open(p) as f:
                n = sum(1 for _ in f)
            print(f"  {split}: {n} frames")
        else:
            print(f"  {split}: NOT FOUND")

# ── Combine per-split manifests into all_manifest.jsonl ─────────────────────
# Annotation pipeline needs a single combined manifest so all splits are
# annotated in ONE run (per-split runs overwrite each other's output).
ALL_MANIFEST = f"{UNIFIED_OUTPUT}/all_manifest.jsonl"
if not os.path.exists(ALL_MANIFEST):
    print("\nCombining split manifests → all_manifest.jsonl...")
    combined = []
    for split in ("train", "val", "test"):
        p = f"{UNIFIED_OUTPUT}/{split}_manifest.jsonl"
        if os.path.exists(p):
            with open(p) as f:
                for line in f:
                    frame = json.loads(line)
                    frame["split"] = split
                    combined.append(frame)
    with open(ALL_MANIFEST, "w") as f:
        for frame in combined:
            f.write(json.dumps(frame) + "\n")
    print(f"  ✓ Combined {len(combined)} frames → all_manifest.jsonl")
else:
    with open(ALL_MANIFEST) as f:
        n = sum(1 for _ in f)
    print(f"✓ all_manifest.jsonl already exists ({n} frames)")

## 6. LLM Annotation

Generates structured JSON annotations (bbox, hazard class, severity, reasoning, action).
~30% of frames with hazards receive LLM counterfactual augmentation.

**Mock mode** (default, free): placeholder annotations for pipeline testing.
**Real mode** (~$5–8): store your Anthropic API key in Colab Secrets → "ANTHROPIC_API_KEY",
then uncomment Option B below.

Per-frame cache enables safe resume if the session disconnects mid-annotation.

In [ ]:
import os

UNIFIED_OUTPUT    = f"{OUTPUTS_ROOT}/data/unified"
ANNOTATION_OUTPUT = f"{OUTPUTS_ROOT}/data/annotated"
ALL_MANIFEST      = f"{UNIFIED_OUTPUT}/all_manifest.jsonl"
os.makedirs(ANNOTATION_OUTPUT, exist_ok=True)

if os.path.exists(f"{ANNOTATION_OUTPUT}/annotated_manifest.json"):
    print("✓ Annotation already complete — skipping.")
else:
    # OPTION A: Mock mode (default, free — no API key needed)
    !python scripts/run_annotation_pipeline.py \
        --manifest {ALL_MANIFEST} \
        --output-dir {ANNOTATION_OUTPUT} \
        --mock-llm \
        2>&1

    # OPTION B: Real annotations — uncomment and set API key in Colab Secrets
    # try:
    #     from google.colab import userdata
    #     os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    # except Exception:
    #     pass  # Key not in secrets — set manually below
    # # os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
    # !python scripts/run_annotation_pipeline.py \
    #     --manifest {ALL_MANIFEST} \
    #     --output-dir {ANNOTATION_OUTPUT} \
    #     2>&1

    if os.path.exists(f"{ANNOTATION_OUTPUT}/annotated_manifest.json"):
        print(f"✓ Annotation complete → {ANNOTATION_OUTPUT}/annotated_manifest.json")
    else:
        print("⚠ annotated_manifest.json not found — check output above")

## 7. Fix Image Paths & Create SFT Files

The unified builder doesn't always carry over `exported_image_path` from the filtered
metadata. This cell maps `frame_id → image_path` from `metadata.json` and writes
`sft_train/val/test.jsonl` with correct paths for each split.

In [ ]:
import json, os

ANNOTATION_OUTPUT = f"{OUTPUTS_ROOT}/data/annotated"
SFT_OUTPUT        = f"{OUTPUTS_ROOT}/data/sft_ready"
FILTER_META       = f"{OUTPUTS_ROOT}/data/nuscenes_filtered/metadata.json"
IMAGES_DIR        = f"{OUTPUTS_ROOT}/data/nuscenes_filtered/images"

os.makedirs(SFT_OUTPUT, exist_ok=True)

if os.path.exists(f"{SFT_OUTPUT}/sft_train.jsonl"):
    print("✓ SFT files already created — skipping.")
else:
    # ── Build frame_id → image_path mapping from rarity-filter metadata ─────
    id_to_image = {}
    if os.path.exists(FILTER_META):
        with open(FILTER_META) as f:
            filtered = json.load(f)
        for frame in filtered:
            fid = frame.get("sample_token", "")
            img = frame.get("exported_image_path", "")
            if img and not os.path.isabs(img):
                img = f"{OUTPUTS_ROOT}/data/nuscenes_filtered/{img}"
            if not img or not os.path.exists(img):
                cam_path = frame.get("cam_front_path", "")
                if cam_path:
                    candidate = f"{IMAGES_DIR}/{os.path.basename(cam_path)}"
                    if os.path.exists(candidate):
                        img = candidate
            id_to_image[fid] = img
        filled = sum(1 for v in id_to_image.values() if v)
        print(f"Image mappings: {filled}/{len(id_to_image)} frames have paths")
    else:
        print(f"⚠ {FILTER_META} not found — image paths will be empty")

    # ── Load annotated manifest ──────────────────────────────────────────────
    ann_path = f"{ANNOTATION_OUTPUT}/annotated_manifest.json"
    if not os.path.exists(ann_path):
        print(f"⚠ {ann_path} not found — run annotation cell first.")
    else:
        with open(ann_path) as f:
            annotated = json.load(f)

        SYSTEM_PROMPT = (
            "You are DriveSense, an autonomous vehicle hazard detection system. "
            "Analyze the dashcam image and identify all safety-critical hazards. "
            "Output a structured JSON response with bounding boxes (normalized 0-1000), "
            "hazard classification, severity assessment, reasoning, and recommended action."
        )
        USER_PROMPT = (
            "Analyze this dashcam image for safety hazards. Identify all hazards with "
            "bounding boxes, classify each hazard, assess severity, explain your reasoning, "
            "and recommend an action. Respond with JSON only."
        )

        # ── Write per-split SFT JSONL files ─────────────────────────────────
        for split in ("train", "val", "test"):
            split_frames = [
                f for f in annotated
                if f.get("split") == split and f.get("annotations")
            ]
            sft_path = f"{SFT_OUTPUT}/sft_{split}.jsonl"
            with open(sft_path, "w") as out:
                for frame in split_frames:
                    fid     = frame.get("frame_id", "")
                    img_path = id_to_image.get(fid, frame.get("image_path", ""))
                    example = {
                        "messages": [
                            {"role": "system", "content": SYSTEM_PROMPT},
                            {"role": "user", "content": [
                                {"type": "image", "image": img_path},
                                {"type": "text",  "text": USER_PROMPT},
                            ]},
                            {"role": "assistant", "content": json.dumps(frame["annotations"])},
                        ],
                        "frame_id": fid,
                        "source":   frame.get("source", ""),
                        "split":    split,
                    }
                    out.write(json.dumps(example) + "\n")
            count = sum(1 for _ in open(sft_path))
            size  = os.path.getsize(sft_path) / 1024
            print(f"  ✓ sft_{split}.jsonl: {count} examples ({size:.1f} KB)")

        # ── Verify image path on first example ──────────────────────────────
        train_path = f"{SFT_OUTPUT}/sft_train.jsonl"
        if os.path.exists(train_path) and os.path.getsize(train_path) > 0:
            with open(train_path) as f:
                ex = json.loads(f.readline())
            for msg in ex.get("messages", []):
                for item in (msg.get("content") if isinstance(msg.get("content"), list) else []):
                    if item.get("type") == "image":
                        p = item.get("image", "")
                        print(f"\nFirst example image: {p}")
                        print(f"Path exists: {os.path.exists(p)}")

## 8. Verification

Final check of all pipeline outputs.

In [ ]:
import os, json

SFT_DIR     = f"{OUTPUTS_ROOT}/data/sft_ready"
FILTER_OUT  = f"{OUTPUTS_ROOT}/data/nuscenes_filtered"
UNIFIED_OUT = f"{OUTPUTS_ROOT}/data/unified"
ANNOTATED   = f"{OUTPUTS_ROOT}/data/annotated"

checks = {
    "nuScenes metadata.json":    f"{FILTER_OUT}/metadata.json",
    "nuScenes metadata.jsonl":   f"{FILTER_OUT}/metadata.jsonl",
    "Spark processed":           f"{OUTPUTS_ROOT}/data/spark_processed",
    "Unified train_manifest":    f"{UNIFIED_OUT}/train_manifest.jsonl",
    "Unified all_manifest":      f"{UNIFIED_OUT}/all_manifest.jsonl",
    "Annotated manifest":        f"{ANNOTATED}/annotated_manifest.json",
    "SFT train":                 f"{SFT_DIR}/sft_train.jsonl",
    "SFT val":                   f"{SFT_DIR}/sft_val.jsonl",
    "SFT test":                  f"{SFT_DIR}/sft_test.jsonl",
}

print("=" * 56)
print("  Pipeline Verification")
print("=" * 56)
all_ok = True
for label, path in checks.items():
    ok = os.path.exists(path)
    if ok and path.endswith(".jsonl"):
        with open(path) as f:
            n = sum(1 for _ in f)
        detail = f" ({n} records)"
    elif ok and path.endswith(".json"):
        detail = f" ({os.path.getsize(path)//1024} KB)"
    elif ok:
        detail = ""
    else:
        detail = ""
    print(f"  {'✓' if ok else '✗'}  {label}{detail}")
    all_ok = all_ok and ok
print("=" * 56)

if all_ok:
    print("\n✓ All outputs present — ready for Notebook 01 (Training)")
    train_path = f"{SFT_DIR}/sft_train.jsonl"
    if os.path.getsize(train_path) > 0:
        with open(train_path) as f:
            ex = json.loads(f.readline())
        print(f"\nSample keys: {list(ex.keys())}")
        if "messages" in ex:
            print(f"Roles: {[m.get('role') for m in ex['messages']]}")
else:
    print("\n⚠ Some outputs missing — rerun the failed cells above.")
    print("  Each step is idempotent (safe to rerun).")